In [1]:
import os
from pathlib import Path
import duckdb

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(ROOT)
con = duckdb.connect()

def load_queries(path):
    text = Path(path).read_text(encoding="utf-8")
    queries, current_name, current_sql = {}, None, []
    for line in text.splitlines():
        if line.strip().startswith("-- name:"):
            if current_name is not None:
                queries[current_name] = "\n".join(current_sql).strip()
            current_name = line.split("-- name:", 1)[1].strip()
            current_sql = []
        else:
            current_sql.append(line)
    if current_name is not None:
        queries[current_name] = "\n".join(current_sql).strip()
    return queries

def run(queries, name):
    return con.execute(queries[name]).df()

In [2]:
for f in ["sql/00_setup.sql", "sql/01_scope.sql", "sql/01_clean.sql", "sql/02_features.sql"]:
    for name, sql in load_queries(f).items():
        if name.startswith("create_"):
            con.execute(sql)

feat = load_queries("sql/02_features.sql")
display(run(feat, "features_reconcile").T)

,0
n_rows,20405666
pu_zone_null,0
pu_borough_null,0
min_hour,0
max_hour,23
min_isodow,1
max_isodow,7
n_pickup_zones,262
min_request_date,2024-12-31 00:00:00
max_request_date,2025-02-01 00:00:00


In [3]:
import pandas as pd
h = load_queries("sql/02d_hotspots.sql"); z = load_queries("sql/02d2_hotspots_zone_hour.sql"); d = load_queries("sql/02d3_hotspots_zone_daypart.sql")
for tbl, build in [("zhw_hotspots", [h["create_zhw_cells"], h["create_zhw_hotspots"]]),
                   ("zh_hotspots", [z["create_zh_cells"], z["create_zh_hotspots"]]),
                   ("coverage_cells", [z["create_coverage_cells"]])]:
    try:
        con.execute(f"SELECT 1 FROM {tbl} LIMIT 1")
    except Exception:
        for stmt in build:
            con.execute(stmt)
for name in ["create_zdd_cells", "create_zdd_hotspots", "create_zdd_zone_severity"]:
    con.execute(d[name])
display(run(d, "zdd_cells_reconcile").T)

,0
n_cells,2579.0
n_zones,260.0
n_slots,10.0
empty_cells,21.0
cells_with_no_valid_wait,0.0
n_scoped,20404892.0
n_valid,20205118.0
n_excluded,199774.0
n_days_mon_fri,23.0
n_days_sat_sun,8.0


In [4]:
cov = run(d, "coverage_tradeoff_v2"); display(cov)
cb = run(d, "coverage_tradeoff_by_borough_v2")
display(cb.pivot(index="design", columns="pu_borough", values="pct_valid_rows_in_adequate_cells"))
display(run(d, "zdd_borough_summary").T)

,design,n_slots,n_cells,n_adequate_cells,zones_with_adequate_cell,total_valid_rows,valid_rows_in_adequate_cells,pct_valid_rows_in_adequate_cells
0,A zone x weekday x hour,168,42842,5221,113,20205118.0,7914749.0,39.17
1,B zone x hour,24,6180,4516,221,20205118.0,19496072.0,96.49
2,C zone x daypart x day type,10,2579,2177,241,20205118.0,20038534.0,99.18
3,D zone x daypart,5,1296,1186,243,20205118.0,20167469.0,99.81
4,E zone (all hours),1,260,250,250,20205118.0,20201782.0,99.98


pu_borough,Bronx,Brooklyn,Manhattan,Queens,Staten Island
design,,,,,
A zone x weekday x hour,6.82,41.10,53.10,33.76,0.00
B zone x hour,96.43,97.45,98.82,94.65,46.40
C zone x daypart x day type,99.15,99.49,99.76,98.79,84.21
D zone x daypart,99.55,99.92,99.92,99.76,98.31
E zone (all hours),100.00,99.99,100.00,99.95,99.85


,0,1,2,3,4,5
pu_borough,ALL,Staten Island,Queens,Brooklyn,Manhattan,Bronx
n_cells,2579,197,686,610,663,423
n_adequate_cells,2177,96,552,554,609,366
total_valid_rows,20205118.0,303909.0,4223481.0,5407640.0,7704238.0,2565850.0
valid_rows_in_adequate_cells,20038534.0,255925.0,4172283.0,5380178.0,7686131.0,2544017.0
pct_valid_rows_in_adequate_cells,99.18,84.21,98.79,99.49,99.76,99.15
total_long_waits,1051522.0,38429.0,334141.0,263033.0,274822.0,141097.0
long_waits_in_adequate_cells,1031391.0,28968.0,328559.0,261771.0,273295.0,138798.0
pct_long_waits_in_adequate_cells,98.09,75.38,98.33,99.52,99.44,98.37
n_hotspot_cells,133,53,51,12,10,7


In [5]:
display(run(d, "zdd_hotspot_summary").T)
display(run(d, "zdd_hotspot_by_slot"))
display(run(d, "zdd_staten_island"))
display(run(d, "zone_severity_summary"))
display(run(d, "zone_severity_top"))
display(run(d, "zdd_vs_zone_hour").T)

,0
n_cells_with_valid_wait,2579
n_adequate_cells,2177
n_zones_with_adequate_cell,241
n_hotspot_1_5x,262
n_hotspot_2x,133
n_hotspot_3x,47
n_hotspot_robust_2x,133
n_time_specific,39
n_persistent,94
n_hotspot_in_airport_zones,12


,daypart,day_type,n_adequate_cells,n_hotspot_cells,n_time_specific,slot_pct_wait_ge10,slot_pct_wait_ge10_ex_airports,pct_excluded
0,1_overnight_0-5,Mon-Fri,216,11,8,9.231,8.661,1.789
1,1_overnight_0-5,Sat-Sun,183,6,3,6.322,6.096,0.739
2,2_morning_6-9,Mon-Fri,236,24,12,6.990,6.924,2.134
3,2_morning_6-9,Sat-Sun,192,9,1,3.427,3.087,1.448
4,3_midday_10-15,Mon-Fri,241,21,0,3.661,3.360,1.447
5,3_midday_10-15,Sat-Sun,225,14,1,3.034,2.726,0.514
6,4_evening_peak_16-19,Mon-Fri,237,13,3,4.104,3.611,0.476
7,4_evening_peak_16-19,Sat-Sun,211,9,1,4.367,3.870,0.312
8,5_late_evening_20-23,Mon-Fri,232,17,4,5.499,4.139,0.183
9,5_late_evening_20-23,Sat-Sun,204,9,6,7.452,6.116,0.156


,PULocationID,pu_zone,n_valid,pct_excluded,n_adequate_cells,zone_pct_wait_ge10,ratio_vs_city,zone_severity_flag,n_hotspot_cells,n_time_specific_cells
0,206,Saint George/New Brighton,41362.0,1.138,10,11.542,2.22,True,6,3
1,23,Bloomfield/Emerson Hill,31210.0,1.611,9,13.627,2.62,True,8,2
2,221,Stapleton,28517.0,1.591,9,11.432,2.20,True,4,2
3,156,Mariners Harbor,27750.0,1.368,8,9.686,1.86,False,3,1
4,118,Heartland Village/Todt Hill,20154.0,1.688,6,14.161,2.72,True,5,0
5,214,South Beach/Dongan Hills,17477.0,1.881,5,10.185,1.96,False,1,0
6,115,Grymes Hill/Clifton,16851.0,1.669,5,14.118,2.71,True,3,0
7,245,West Brighton,15721.0,1.738,6,12.251,2.35,True,3,1
8,251,Westerleigh,15493.0,1.750,6,9.049,1.74,False,1,0
9,187,Port Richmond,14876.0,1.090,5,11.112,2.14,True,2,0


,pu_borough,n_zones,n_zones_eligible,n_zones_severe,pct_valid_rows_in_severe_zones,pct_long_waits_in_severe_zones,city_pct_wait_ge10
0,Staten Island,20,18,14,76.05,81.47,5.204
1,Queens,69,64,7,20.60,50.08,5.204
2,Manhattan,67,66,2,0.19,0.69,5.204
3,Bronx,43,42,1,0.11,0.69,5.204
4,Brooklyn,61,60,1,0.02,0.06,5.204


,pu_borough,pu_zone,n_valid,pct_excluded,zone_pct_wait_ge10,zone_pct_wait_ge10_wilson_low,ratio_vs_city,zone_severity_flag,n_adequate_cells,n_hotspot_cells,n_time_specific_cells
0,Bronx,City Island,2803.0,3.445,34.784,33.043,6.68,True,0,0,0
1,Queens,JFK Airport,366295.0,0.066,26.725,26.582,5.14,True,10,10,1
2,Queens,Rockaway Park,11330.0,3.146,23.972,23.195,4.61,True,4,4,0
3,Queens,Hammels/Arverne,31820.0,2.234,23.755,23.291,4.56,True,10,10,0
4,Staten Island,Charleston/Tottenville,8598.0,2.173,21.563,20.707,4.14,True,3,3,0
5,Manhattan,Randalls Island,2279.0,3.021,21.062,19.437,4.05,True,0,0,0
6,Queens,Far Rockaway,46519.0,2.160,20.716,20.350,3.98,True,10,10,0
7,Staten Island,Great Kills,13808.0,2.279,16.766,16.152,3.22,True,6,5,1
8,Staten Island,Eltingville/Annadale/Prince's Bay,9163.0,2.334,16.065,15.327,3.09,True,3,2,0
9,Staten Island,Rossville/Woodrow,5243.0,3.194,15.735,14.775,3.02,True,2,1,0


,0
zones_flagged_in_both,34
zones_only_in_zone_hour,16
zones_only_in_daypart,22


In [6]:
out = Path("dashboard/data"); out.mkdir(parents=True, exist_ok=True)
con.execute("SELECT * FROM zdd_hotspots ORDER BY PULocationID, daypart, day_type").df().to_csv(out / "2d3_zone_daypart_hotspots.csv", index=False)
con.execute("SELECT * FROM zdd_zone_severity ORDER BY PULocationID").df().to_csv(out / "2d3_zone_severity.csv", index=False)
cov.to_csv(out / "2d3_coverage_tradeoff_v2.csv", index=False)

In [7]:
for name, title in [("zdd_top_by_rate", "Rate: lower end of interval"),
                    ("zdd_top_non_airport_by_rate", "Rate, airport zones hidden (display aid)"),
                    ("zdd_top_by_impact", "Impact: number of long waits"),
                    ("zdd_top_by_excess", "Excess long waits above slot baseline")]:
    print(title); display(run(d, name))

Rate: lower end of interval


,pu_borough,pu_zone,daypart,day_type,n_valid,pct_excluded,avg_trips_per_day,pct_wait_ge10,pct_wait_ge10_wilson_low,pct_ge10_if_excluded_all_short,slot_pct_wait_ge10,ratio_vs_slot,zone_pct_wait_ge10_all_slots,excess_vs_zone_pts,hotspot_type,p50_min,p90_min,pct_uber_of_valid,hotspot_robust_flag
0,Queens,JFK Airport,5_late_evening_20-23,Sat-Sun,28327,0.014,3541.4,42.631,42.056,42.625,7.452,5.72,26.725,15.906,time-specific,9.117,17.183,78.6,True
1,Queens,JFK Airport,5_late_evening_20-23,Mon-Fri,72524,0.045,3154.7,36.052,35.703,36.035,5.499,6.56,26.725,9.327,persistent,8.433,14.883,79.0,True
2,Staten Island,Great Kills,1_overnight_0-5,Mon-Fri,1038,5.119,47.6,37.861,34.960,35.923,9.231,4.10,16.766,21.096,time-specific,8.333,17.687,82.6,True
3,Queens,Hammels/Arverne,1_overnight_0-5,Mon-Fri,2637,2.152,117.2,36.405,34.590,35.622,9.231,3.94,23.755,12.650,persistent,8.000,17.807,76.7,True
4,Queens,Hammels/Arverne,2_morning_6-9,Mon-Fri,5923,5.126,271.4,33.361,32.172,31.651,6.990,4.77,23.755,9.606,persistent,7.683,15.983,74.6,True
5,Queens,Hammels/Arverne,5_late_evening_20-23,Sat-Sun,1480,0.404,185.8,33.446,31.088,33.311,7.452,4.49,23.755,9.690,persistent,7.592,16.258,82.2,True
6,Queens,JFK Airport,1_overnight_0-5,Mon-Fri,30905,0.091,1344.9,31.273,30.759,31.245,9.231,3.39,26.725,4.548,persistent,7.817,15.533,79.2,True
7,Queens,Far Rockaway,1_overnight_0-5,Mon-Fri,3386,2.813,151.5,32.103,30.551,31.200,9.231,3.48,20.716,11.387,persistent,7.183,17.017,76.3,True
8,Staten Island,Bloomfield/Emerson Hill,1_overnight_0-5,Mon-Fri,2259,3.130,101.4,31.784,29.896,30.789,9.231,3.44,13.627,18.157,time-specific,7.700,15.267,81.5,True
9,Queens,Rockaway Park,2_morning_6-9,Mon-Fri,1547,6.356,71.8,31.739,29.467,29.722,6.990,4.54,23.972,7.767,persistent,7.167,14.800,78.2,True


Rate, airport zones hidden (display aid)


,pu_borough,pu_zone,daypart,day_type,n_valid,pct_excluded,avg_trips_per_day,pct_wait_ge10,pct_wait_ge10_wilson_low,pct_ge10_if_excluded_all_short,slot_pct_wait_ge10,ratio_vs_slot,zone_pct_wait_ge10_all_slots,excess_vs_zone_pts,hotspot_type,p50_min,p90_min,pct_uber_of_valid,hotspot_robust_flag
0,Staten Island,Great Kills,1_overnight_0-5,Mon-Fri,1038,5.119,47.6,37.861,34.960,35.923,9.231,4.10,16.766,21.096,time-specific,8.333,17.687,82.6,True
1,Queens,Hammels/Arverne,1_overnight_0-5,Mon-Fri,2637,2.152,117.2,36.405,34.590,35.622,9.231,3.94,23.755,12.650,persistent,8.000,17.807,76.7,True
2,Queens,Hammels/Arverne,2_morning_6-9,Mon-Fri,5923,5.126,271.4,33.361,32.172,31.651,6.990,4.77,23.755,9.606,persistent,7.683,15.983,74.6,True
3,Queens,Hammels/Arverne,5_late_evening_20-23,Sat-Sun,1480,0.404,185.8,33.446,31.088,33.311,7.452,4.49,23.755,9.690,persistent,7.592,16.258,82.2,True
4,Queens,Far Rockaway,1_overnight_0-5,Mon-Fri,3386,2.813,151.5,32.103,30.551,31.200,9.231,3.48,20.716,11.387,persistent,7.183,17.017,76.3,True
5,Staten Island,Bloomfield/Emerson Hill,1_overnight_0-5,Mon-Fri,2259,3.130,101.4,31.784,29.896,30.789,9.231,3.44,13.627,18.157,time-specific,7.700,15.267,81.5,True
6,Queens,Rockaway Park,2_morning_6-9,Mon-Fri,1547,6.356,71.8,31.739,29.467,29.722,6.990,4.54,23.972,7.767,persistent,7.167,14.800,78.2,True
7,Brooklyn,Brooklyn Navy Yard,1_overnight_0-5,Mon-Fri,2291,0.650,100.3,31.034,29.173,30.833,9.231,3.36,8.952,22.082,time-specific,6.750,18.250,71.7,True
8,Staten Island,Bloomfield/Emerson Hill,5_late_evening_20-23,Sat-Sun,1181,0.338,148.1,31.583,28.996,31.477,7.452,4.24,13.627,17.956,time-specific,7.533,15.733,83.5,True
9,Staten Island,Saint George/New Brighton,1_overnight_0-5,Mon-Fri,2779,1.138,122.2,28.427,26.781,28.104,9.231,3.08,11.542,16.885,time-specific,7.133,14.650,76.1,True


Impact: number of long waits


,pu_borough,pu_zone,daypart,day_type,n_valid,n_wait_ge10,pct_excluded,avg_trips_per_day,pct_wait_ge10,slot_pct_wait_ge10,ratio_vs_slot,hotspot_flag,hotspot_type
0,Queens,JFK Airport,5_late_evening_20-23,Mon-Fri,72524,26146,0.045,3154.7,36.052,5.499,6.56,True,persistent
1,Queens,LaGuardia Airport,5_late_evening_20-23,Mon-Fri,77779,19450,0.187,3388.0,25.007,5.499,4.55,True,time-specific
2,Queens,JFK Airport,4_evening_peak_16-19,Mon-Fri,62769,16377,0.076,2731.2,26.091,4.104,6.36,True,persistent
3,Queens,JFK Airport,3_midday_10-15,Mon-Fri,67577,12738,0.068,2940.1,18.850,3.661,5.15,True,persistent
4,Queens,JFK Airport,5_late_evening_20-23,Sat-Sun,28327,12076,0.014,3541.4,42.631,7.452,5.72,True,time-specific
5,Queens,JFK Airport,1_overnight_0-5,Mon-Fri,30905,9665,0.091,1344.9,31.273,9.231,3.39,True,persistent
6,Manhattan,Times Sq/Theatre District,5_late_evening_20-23,Mon-Fri,58373,8115,0.545,2551.9,13.902,5.499,2.53,True,time-specific
7,Manhattan,Midtown Center,5_late_evening_20-23,Mon-Fri,60739,7851,0.164,2645.2,12.926,5.499,2.35,True,time-specific
8,Queens,LaGuardia Airport,5_late_evening_20-23,Sat-Sun,22093,6096,0.154,2765.9,27.592,7.452,3.70,True,time-specific
9,Queens,JFK Airport,4_evening_peak_16-19,Sat-Sun,23145,6046,0.030,2894.0,26.122,4.367,5.98,True,persistent


Excess long waits above slot baseline


,pu_borough,pu_zone,daypart,day_type,n_valid,n_wait_ge10,excess_long_waits_vs_slot,excess_long_waits_vs_zone,pct_excluded,pct_wait_ge10,slot_pct_wait_ge10,ratio_vs_slot,hotspot_type
0,Queens,JFK Airport,5_late_evening_20-23,Mon-Fri,72524,26146,22158.1,6764.0,0.045,36.052,5.499,6.56,persistent
1,Queens,LaGuardia Airport,5_late_evening_20-23,Mon-Fri,77779,19450,15173.1,10156.8,0.187,25.007,5.499,4.55,time-specific
2,Queens,JFK Airport,4_evening_peak_16-19,Mon-Fri,62769,16377,13801.1,-398.0,0.076,26.091,4.104,6.36,persistent
3,Queens,JFK Airport,3_midday_10-15,Mon-Fri,67577,12738,10264.2,-5321.9,0.068,18.850,3.661,5.15,persistent
4,Queens,JFK Airport,5_late_evening_20-23,Sat-Sun,28327,12076,9965.1,4505.6,0.014,42.631,7.452,5.72,time-specific
5,Queens,JFK Airport,1_overnight_0-5,Mon-Fri,30905,9665,6812.1,1405.7,0.091,31.273,9.231,3.39,persistent
6,Queens,JFK Airport,4_evening_peak_16-19,Sat-Sun,23145,6046,5035.2,-139.5,0.030,26.122,4.367,5.98,persistent
7,Manhattan,Times Sq/Theatre District,5_late_evening_20-23,Mon-Fri,58373,8115,4905.2,4011.2,0.545,13.902,5.499,2.53,time-specific
8,Manhattan,Midtown Center,5_late_evening_20-23,Mon-Fri,60739,7851,4511.1,4241.0,0.164,12.926,5.499,2.35,time-specific
9,Queens,LaGuardia Airport,5_late_evening_20-23,Sat-Sun,22093,6096,4449.6,3456.3,0.154,27.592,7.452,3.70,time-specific


In [8]:
out = Path("dashboard/data"); out.mkdir(parents=True, exist_ok=True)
con.execute("SELECT * FROM zdd_hotspots ORDER BY PULocationID, daypart, day_type").df().to_csv(out / "2d3_zone_daypart_hotspots.csv", index=False)
con.execute("SELECT * FROM zdd_zone_severity ORDER BY PULocationID").df().to_csv(out / "2d3_zone_severity.csv", index=False)
cov.to_csv(out / "2d3_coverage_tradeoff_v2.csv", index=False)